# 📊 Day 9 — Hypothesis Testing: Intuition & Null Hypothesis
**90-Day Data Science Roadmap | FC Lahore Lions Dataset**

> **Core question:** Is what we're seeing *real*, or is it just *luck*?

---
**Concepts covered:**
- H0 (Null) vs H1 (Alternative) Hypothesis
- p-value — what it actually means
- Significance level (alpha)
- Type I & Type II Errors
- Reject vs Fail to Reject H0
- Sample size effect on p-value
- p-hacking / Multiple Comparisons trap

## 📦 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

np.random.seed(42)  # same seed convention from Days 1-8
print('Libraries loaded ✅')

---
## ⚽ Section 1 — The Story (FC Lahore Lions, Penalty Taker)

Our penalty taker has always scored **70%** of penalties historically.  
This season, after working with a **new coach**, we want to know: did the coaching *genuinely* improve him, or was any improvement just a lucky streak?

| | Claim |
|---|---|
| **H0** | "The penalty success rate didn't really change — any difference is just random luck." |
| **H1** | "The penalty success rate genuinely improved after the new coach." |

In [ ]:
# --- Generate synthetic FC Lahore Lions penalty data ---
# Before new coach: 30 penalties, historical 70% success rate
penalties_before = np.random.binomial(1, 0.70, 30)

# After new coach: 30 penalties, true rate secretly improved to 78%
# (we'll see if the test can DETECT this real improvement)
penalties_after = np.random.binomial(1, 0.78, 30)

rate_before = penalties_before.mean()
rate_after  = penalties_after.mean()

print(f'Before new coach — success rate: {rate_before:.4f} ({rate_before*100:.1f}%)')
print(f'After new coach  — success rate: {rate_after:.4f} ({rate_after*100:.1f}%)')
print(f'Observed difference: {(rate_after - rate_before)*100:.1f}%')

> 👀 **Notice:** With seed=42, 'After' actually looks *lower* than 'Before' in raw numbers. This is exactly why we can't just eyeball data — small samples (30 penalties) are noisy. A few kicks in either direction flip the numbers. We need the test.

In [ ]:
# --- Visualise: Before vs After success rates ---
fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(['Before Coach', 'After Coach'],
              [rate_before, rate_after],
              color=['#4C72B0', '#55A868'],
              width=0.4, edgecolor='white')

ax.axhline(0.70, color='gray', linestyle='--', linewidth=1.2, label='Historical baseline (70%)')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Penalty Success Rate')
ax.set_title('FC Lahore Lions — Penalty Success Rate: Before vs After New Coach')
ax.legend()

for bar, rate in zip(bars, [rate_before, rate_after]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{rate*100:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('day09_penalty_rates.png', dpi=120)
plt.show()
print('Chart saved as day09_penalty_rates.png')

---
## 🧪 Section 2 — Running the Hypothesis Test

We use a **two-sample t-test** to compare the before and after groups.

**Decision rule (alpha = 0.05):**
- `p-value ≤ 0.05` → Reject H0 (statistically significant, probably real)
- `p-value > 0.05` → Fail to reject H0 (not enough evidence)

In [ ]:
# --- Two-sample t-test ---
t_stat, p_value = stats.ttest_ind(penalties_before, penalties_after)

alpha = 0.05

print('='*50)
print(f't-statistic : {t_stat:.6f}')
print(f'p-value     : {p_value:.6f}')
print(f'alpha       : {alpha}')
print('='*50)

if p_value <= alpha:
    print('DECISION: Reject H0')
    print('MEANING : Strong evidence of a real change — probably not luck.')
else:
    print('DECISION: Fail to reject H0')
    print('MEANING : Not enough evidence of a real change — could be luck.')

### 🔍 Reading the p-value

```
p = 0.77 means:
"If H0 were true (no real change), a result like this would happen 77% of the time by luck."
77% is not surprising at all → luck still explains this → we stick with H0.
```

**Real lesson hidden here:**  
We secretly generated the data with a genuine 8% improvement (70% → 78%).  
But with only **30 penalties**, the noise drowned out the signal — the test couldn't detect it.  
→ This is called **low statistical power**: a true effect can fail to reach significance when your sample is too small.

---
## 🔢 Section 3 — Type I & Type II Errors (Visualised)

In [ ]:
# --- Error Types: 2x2 grid ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

table_data = [
    ['', 'Reality: H0 TRUE\n(no real effect)', 'Reality: H1 TRUE\n(real effect exists)'],
    ['You say:\nReject H0',
     '❌ TYPE I ERROR\n(False Alarm)\n"Said real, was luck"',
     '✅ CORRECT\n"True Positive"'],
    ['You say:\nFail to Reject H0',
     '✅ CORRECT\n"True Negative"',
     '❌ TYPE II ERROR\n(Missed It)\n"Said luck, was real"'],
]

colors = [
    ['#2c2c2c', '#2c2c2c', '#2c2c2c'],
    ['#2c2c2c', '#8B1A1A', '#1A5C1A'],
    ['#2c2c2c', '#1A5C1A', '#8B1A1A'],
]

table = ax.table(cellText=table_data,
                 cellLoc='center',
                 loc='center',
                 cellColours=colors)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 3.2)

for (row, col), cell in table.get_celld().items():
    cell.set_text_props(color='white', fontweight='bold' if row == 0 or col == 0 else 'normal')
    cell.set_edgecolor('#555555')

ax.set_title('Type I vs Type II Error', fontsize=13, fontweight='bold', pad=15)
fig.patch.set_facecolor('#1a1a1a')
plt.tight_layout()
plt.savefig('day09_error_types.png', dpi=120, facecolor='#1a1a1a')
plt.show()
print('Chart saved as day09_error_types.png')

---
## 📈 Section 4 — Sample Size Effect on p-value

**Key question:** What happens to the p-value when we increase sample size, with the same real 8% improvement baked in?

In [ ]:
# --- p-value vs sample size ---
sample_sizes = [10, 20, 30, 50, 100, 200, 500, 1000, 2000, 5000]
p_values     = []

np.random.seed(42)
for n in sample_sizes:
    before = np.random.binomial(1, 0.70, n)
    after  = np.random.binomial(1, 0.78, n)  # true improvement: 70% → 78%
    _, p   = stats.ttest_ind(before, after)
    p_values.append(p)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sample_sizes, p_values, marker='o', color='#4C72B0', linewidth=2)
ax.axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='alpha = 0.05 cutoff')
ax.fill_between(sample_sizes, p_values, 0.05,
                where=[p > 0.05 for p in p_values],
                alpha=0.15, color='orange', label='Fail to reject H0 zone')
ax.fill_between(sample_sizes, p_values, 0.05,
                where=[p <= 0.05 for p in p_values],
                alpha=0.15, color='green', label='Reject H0 zone')

ax.set_xscale('log')
ax.set_xlabel('Sample Size (n)')
ax.set_ylabel('p-value')
ax.set_title('p-value vs Sample Size\n(Same 8% real improvement baked in throughout)')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('day09_sample_size_effect.png', dpi=120)
plt.show()

print('\nSample size → p-value table:')
print(f'{"n":>6} | {"p-value":>10} | Decision')
print('-'*38)
for n, p in zip(sample_sizes, p_values):
    decision = 'Reject H0 ✅' if p <= 0.05 else 'Fail to Reject H0 ❌'
    print(f'{n:>6} | {p:>10.4f} | {decision}')

### 🔑 Insight
The **same 8% real improvement** goes undetected at n=30, but becomes clearly significant at larger n.  
This means: **a true effect can fail to reach significance if your sample is too small.**

⚠️ It also means: with massive samples (n=5000), even a *trivially small* real difference (0.1%) can become "statistically significant" — which is why **effect size always needs to be checked alongside p-value**.

---
## 🚨 Section 5 — p-hacking Demo (Multiple Comparisons Trap)

**Scenario:** An analyst runs 20 different tests on match data, "to see what sticks."

In [ ]:
# --- Simulate p-hacking ---
# Run 20 tests where H0 is TRUE (no real effect in any of them)
np.random.seed(42)
n_tests   = 20
fake_vars = ['jersey_number', 'boot_brand', 'hair_length', 'shirt_size',
             'preferred_foot', 'warmup_time', 'social_media_followers',
             'agent_nationality', 'lucky_number', 'stadium_row_number',
             'pregame_meal_type', 'sleep_hours', 'playlist_genre',
             'travel_distance', 'hotel_floor', 'jersey_wash_temp',
             'handshake_duration', 'bench_height', 'captain_age',
             'referee_height']

print('Running 20 tests where H0 is TRUE (no real effect in ANY of them)...')
print(f'{"Test Variable":<30} | {"p-value":>8} | Result')
print('-'*65)

false_positives = 0
for var in fake_vars:
    group_a = np.random.normal(0, 1, 30)  # pure noise
    group_b = np.random.normal(0, 1, 30)  # pure noise
    _, p    = stats.ttest_ind(group_a, group_b)
    flag    = '⚠️ SIGNIFICANT (FALSE POSITIVE!)' if p <= 0.05 else ''
    if p <= 0.05:
        false_positives += 1
    print(f'{var:<30} | {p:>8.4f} | {flag}')

print('='*65)
print(f'False positives found: {false_positives} out of {n_tests} tests')
print(f'Expected by chance  : ~{n_tests * 0.05:.1f} (that is: {n_tests} tests × 5% alpha)')
print()
print('⚠️  If you reported ONLY the significant result as your headline finding,')
print('    that would be p-hacking — cherry-picking from random noise.')

### 🔑 Why this matters
Running 20 tests at α=0.05 means you'd expect **~1 false positive** just by probability — even if there's genuinely nothing going on.  
Announcing the one test that hit p<0.05 as a "finding" — without disclosing the other 19 — is misleading.

**Fix:** Pre-register your hypothesis *before* looking at data. Test one specific, planned question.

---
## 🔁 Section 6 — Alpha Comparison (0.05 vs 0.01)

In [ ]:
# --- Analyst A (alpha=0.05) vs Analyst B (alpha=0.01) on the same p-value ---
# From Round 2 of the quiz: same data, same p-value, different alphas = different conclusions

# Simulate a test with moderate signal
np.random.seed(10)
team_a = np.random.normal(1.8, 0.5, 40)  # goals per match, old tactics
team_b = np.random.normal(2.1, 0.5, 40)  # goals per match, new manager

_, shared_p = stats.ttest_ind(team_a, team_b)

print(f'Shared p-value for both analysts: {shared_p:.4f}')
print()
print('Analyst A (alpha = 0.05):')
if shared_p <= 0.05:
    print(f'  {shared_p:.4f} ≤ 0.05 → Reject H0 → "Significant improvement found"')
else:
    print(f'  {shared_p:.4f} > 0.05 → Fail to reject H0 → "Not enough evidence"')

print()
print('Analyst B (alpha = 0.01):')
if shared_p <= 0.01:
    print(f'  {shared_p:.4f} ≤ 0.01 → Reject H0 → "Significant improvement found"')
else:
    print(f'  {shared_p:.4f} > 0.01 → Fail to reject H0 → "Not enough evidence"')

print()
print('Same data. Same p-value. Different conclusions.')
print('Neither is wrong — they just set different bars for "convincing enough" in advance.')

---
## 📌 Day 9 Summary

| Concept | One-liner |
|---|---|
| **H0** | "Nothing changed — it's luck." (default assumption) |
| **H1** | "Something real happened." (what we're testing for) |
| **p-value** | Probability of seeing this result IF H0 were true |
| **alpha** | Cutoff line decided *before* looking at data (usually 0.05) |
| **Reject H0** | p ≤ alpha — strong enough evidence against H0 |
| **Fail to reject H0** | p > alpha — not enough evidence (≠ H0 is proven true) |
| **Type I Error** | False alarm — said "real," was luck |
| **Type II Error** | Missed it — said "luck," was real |
| **Sample size ↑** | p-value tends to ↓ (real effects become detectable) |
| **p-hacking** | Running many tests and reporting only the significant one |

---

### 🚩 Personal revisit flags
- **Evidence vs Proof language** — hypothesis testing gives *strong evidence*, never *100% proof*. Even p=0.002 has a 0.2% chance of being a coincidence.
- **H0/H1 are claims, not numbers** — don't confuse the observed % with the hypothesis statement.

---
*Day 10 → t-tests & z-tests: applying the framework in practice*